In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../'))

import torch
import numpy as np
from wiener_chaos.pricing.options import get_vegas
import pickle
from math import exp
import pandas as pd
from wiener_chaos.pricing.options import _bs_price, implied_vol_bisection

from wiener_chaos.pricing.rough_heston import rh_price
from tqdm import tqdm

In [ ]:
params = (1.5,0.04,0.5,-0.7,0.55,0.04)  # kappa, theta, sigma, rho, alpha, v0

In [ ]:
# Model + market setup
S0 = 100.0
r = 0
q = 0

In [ ]:
maturities_calibr = [0.019178, 0.038356, 0.082192, 0.164274, 0.246411, 0.328548,
              0.492823, 0.684476, 1, 1.5]
strikes_calibr = [np.linspace(0.8*S0, 1.2*S0, 9).tolist() for _ in range(len(maturities_calibr))]

maturities_test = [0.02, 0.05, 0.10, 0.28, 0.40, 0.85, 1.25]
strikes_test = [np.linspace(0.8*S0, 1.2*S0, 9).tolist() for _ in range(len(maturities_test))]

In [ ]:
prices_calibr = []
iv_calibr = []

for t, T in tqdm(enumerate(maturities_calibr), total=len(maturities_calibr), desc="Processing"):
    prices_t, iv_t = [], []
    for k, K in enumerate(strikes_calibr[t]):
        c = rh_price(params, r, q, T, S0, K, flag="call")
        iv_c = implied_vol_bisection(c, S0, K, T, r, q, flag="call")
        prices_t.append(c.item())
        iv_t.append(iv_c)
    prices_calibr.append(prices_t)
    iv_calibr.append(iv_t)

In [ ]:
prices_test = []
iv_test = []

for t, T in tqdm(enumerate(maturities_test), total=len(maturities_test), desc="Processing"):
    prices_t, iv_t = [], []
    for k, K in enumerate(strikes_test[t]):
        c = rh_price(params, r, q, T, S0, K, flag="call")
        iv_c = implied_vol_bisection(c, S0, K, T, r, q, flag="call")
        prices_t.append(c.item())
        iv_t.append(iv_c)
    prices_test.append(prices_t)
    iv_test.append(iv_t)

In [ ]:
data = {}
data["params"] = params

data["maturities_calibr"] = maturities_calibr
data["strikes_calibr"] = strikes_calibr
data["prices_calibr"] = prices_calibr
data["iv_calibr"] = iv_calibr

data["maturities_test"] = maturities_test
data["strikes_test"] = strikes_test
data["prices_test"] = prices_test
data["iv_test"] = iv_test

data["r"] = r
data["q"] = q
data["S0"] = S0

with open("rough_heston_data.pkl", "wb") as f:
    pickle.dump(data, f)

In [ ]:
# ---- Plot Heston IV surface: x=strikes, y=maturities ----
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (needed for 3D projection)

# Build mesh so shapes match iv/prices: (len(strikes), len(maturities))
X, Y = np.meshgrid(np.array(strikes_calibr[0]), np.array(maturities_calibr), indexing="ij")  # X[i,j]=strike_i, Y[i,j]=mat_j

Z = np.array(iv_calibr).T 

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")

surf = ax.plot_surface(X, Y, Z, cmap="viridis", edgecolor="none")

ax.set_xlabel("Strike")
ax.set_ylabel("Maturity")
ax.set_zlabel("Implied Volatility")  # change to "Price" if Z=prices
ax.set_title("Rough Heston Implied Volatility Surface")

cbar = fig.colorbar(surf, shrink=0.6, aspect=12)
cbar.set_label("IV")  # change to "Price" if Z=prices

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
for j, T in enumerate(maturities_calibr):
    plt.plot(
        strikes_calibr[j], iv_calibr[j],
        marker="o", linewidth=1.8, label=f"T = {T:.2f}"
    )

plt.xlabel("Strike")
plt.ylabel("Implied Volatility")
plt.title("Heston IV Smiles by Maturity")
plt.legend(title="Maturity")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()